# Figure: Classifier Accuracy vs Noise Strength $\kappa$

Trains MLP and CNN classifiers on CIFAR-10 with multiplicative Gaussian input noise $x \leftarrow x \cdot (1 + \kappa Z)$, $Z\sim\mathcal{N}(0,1)$, sweeping $\kappa$, and reports clean test accuracy. Also runs the ML-Leaks attack pipeline as a side-channel.

**External deps:** `classifier.py`, `mlLeaks.py` (not yet in this repo — see README §Status). Until those are provided, the cells that import them will fail; the model definitions and data-loading cells run independently.


In [ ]:
# (removed for repo) from google.colab import drive
# (removed for repo) drive.mount(...)

In [ ]:
import os
os.makedirs("../figures", exist_ok=True)


In [ ]:
# Reimplementation of the original Theano/Lasagne classifier.py using PyTorch
%%writefile classifier.py
# - iterate_minibatches works the same (shuffles once per call, yields remainder)
# - model='cnn' | 'nn' | anything-else -> softmax (logistic regression)
# - channels-first expected for CNN: (N, C, H, W)
# - inputs for NN/softmax: (N, D)
# - uses Adam, CrossEntropyLoss, and weight_decay to mimic L2
# - prints Accuracy and classification_report like the original

import sys
sys.dont_write_bytecode = True

import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as tv_models
import torch.nn.functional as F

from sklearn.metrics import precision_score, recall_score


def iterate_minibatches(inputs, targets, batch_size, shuffle=True):
    assert len(inputs) == len(targets)
    if shuffle:
        indices = np.arange(len(inputs))
        np.random.shuffle(indices)

    start_idx = None
    for start_idx in range(0, len(inputs) - batch_size + 1, batch_size):
        if shuffle:
            excerpt = indices[start_idx:start_idx + batch_size]
        else:
            excerpt = slice(start_idx, start_idx + batch_size)
        yield inputs[excerpt], targets[excerpt]

    if start_idx is not None and start_idx + batch_size < len(inputs):
        excerpt = indices[start_idx + batch_size:] if shuffle else slice(start_idx + batch_size, len(inputs))
        yield inputs[excerpt], targets[excerpt]


# ---------------------------
#           Models
# ---------------------------
class CNNNet(nn.Module):
    """
    Mirrors get_cnn_model:
    conv5x5 (pad='same') -> maxpool2 -> conv5x5 (valid) -> maxpool2 -> tanh FC -> softmax(out)
    Note: For training with CrossEntropyLoss we return logits (no softmax in forward).
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        # n_in is the shape tuple of train_x: (N, C, H, W)
        _, C, H, W = n_in
        # conv1: pad='same' for 5x5 -> pad=2
        self.conv1 = nn.Conv2d(C, 32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2)

        # conv2: valid (no padding)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=5, padding=0)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2)

        # compute flatten size after conv/pool stack
        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = self.pool1(self.relu1(self.conv1(dummy)))
            x = self.pool2(self.relu2(self.conv2(x)))
            flat_dim = x.numel()

        self.fc = nn.Linear(flat_dim, n_hidden)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.tanh(self.fc(x))
        logits = self.out(x)  # no softmax here; CrossEntropyLoss expects logits
        return logits

class CIFARResNet18(nn.Module):
    """
    ResNet-18 adapted for CIFAR-10 (32x32):
      - conv1: 3x3, stride=1, padding=1
      - remove initial maxpool
      - fc: num_classes
    """
    def __init__(self, n_in, n_out):
        super().__init__()
        # n_in is (N, C, H, W); we only need num_classes
        base = tv_models.resnet18(weights=None)
        # tweak stem for CIFAR
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        # replace classifier
        base.fc = nn.Linear(base.fc.in_features, n_out)
        self.net = base

    def forward(self, x):
        return self.net(x)  # logits


class CIFARResNet34(nn.Module):
    """
    ResNet-34 adapted for CIFAR-10 (32x32):
      - conv1: 3x3, stride=1, padding=1
      - remove initial maxpool
      - fc: num_classes
    """
    def __init__(self, n_in, n_out):
        super().__init__()
        base = tv_models.resnet34(weights=None)
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        base.fc = nn.Linear(base.fc.in_features, n_out)
        self.net = base

    def forward(self, x):
        return self.net(x)  # logits


# class MLPNet(nn.Module):
#     """
#     Mirrors get_nn_model: Input (N, D) -> Dense(tanh, n_hidden) -> Dense(n_out)
#     """
#     def __init__(self, n_in, n_hidden, n_out):
#         super().__init__()
#         # n_in is train_x.shape: (N, D)
#         D = n_in[1]
#         self.fc = nn.Linear(D, n_hidden)
#         self.tanh = nn.Tanh()
#         self.out = nn.Linear(n_hidden, n_out)

#     def forward(self, x):
#         x = self.tanh(self.fc(x))
#         logits = self.out(x)
#         return logits


# class SoftmaxNet(nn.Module):
#     """
#     Mirrors get_softmax_model: Input (N, D) -> Dense(n_out)
#     """
#     def __init__(self, n_in, n_out):
#         super().__init__()
#         D = n_in[1]
#         self.out = nn.Linear(D, n_out)

#     def forward(self, x):
#         logits = self.out(x)
#         return logits
class MLPNet(nn.Module):
    """
    Mirrors get_nn_model: Input (N, D) -> Dense(tanh, n_hidden) -> Dense(n_out)
    Robust: always flattens input to (N, D) inside forward().
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        # n_in is train_x.shape: (N, D) after preprocessing/flattening in train_model
        D = n_in[1]
        self.fc = nn.Linear(D, n_hidden)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        # Defensive: if x has spatial dims (N, C, H, W) flatten to (N, D)
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        x = self.tanh(self.fc(x))
        logits = self.out(x)
        return logits


class SoftmaxNet(nn.Module):
    """
    Mirrors get_softmax_model: Input (N, D) -> Dense(n_out)
    Robust: always flattens input to (N, D) inside forward().
    """
    def __init__(self, n_in, n_out):
        super().__init__()
        D = n_in[1]
        self.out = nn.Linear(D, n_out)

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        logits = self.out(x)
        return logits


# ---------------------------
#          Training
# ---------------------------
@torch.no_grad()
def _predict_batches(model, inputs, targets, batch_size, device, is_cnn):
    # Returns predicted class indices (numpy)
    model.eval()
    preds = []
    if batch_size > len(targets):
        batch_size = len(targets)
    for xb_np, _ in iterate_minibatches(inputs, targets, batch_size, shuffle=False):
        xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
        if is_cnn and xb.ndim == 4:
            # channels-first expected; no change
            pass
        elif not is_cnn and xb.ndim == 2:
            # fine
            pass
        else:
            raise ValueError("Input shape does not match model type (cnn vs nn/softmax).")
        logits = model(xb)
        pred = torch.argmax(logits, dim=1)
        preds.append(pred.cpu().numpy())
    return np.concatenate(preds, axis=0) if len(preds) > 0 else np.array([], dtype=np.int64)

@torch.no_grad()
def eval_accuracy(model, inputs, targets, batch_size, device, is_cnn) -> float:
    """
    Compute accuracy on (inputs, targets) without modifying training code.
    """
    if inputs is None or targets is None or len(targets) == 0:
        return float('nan')
    preds = _predict_batches(model, inputs, targets, batch_size, device, is_cnn)
    return float(accuracy_score(targets, preds))


def train_model(
    dataset,
    n_hidden=50,
    batch_size=100,
    epochs=100,
    learning_rate=0.01,
    model='cnn',
    l2_ratio=1e-7,
    *,
    mg_kappa: float = 0.0,            # 0.0 disables MG noise (default = old behavior)
    mg_mode: str = "gauss",           # "gauss" or "lognorm"
    target_test_acc= None,            # e.g., 0.35 to stop at 35% test accuracy
    eval_every: int = 1               # evaluate every N epochs for early-stop
):
    """
    dataset: tuple (train_x, train_y, test_x, test_y) with NumPy arrays
      - CNN expects train_x/test_x: (N, C, H, W) channels-first
      - NN/softmax expect train_x/test_x: (N, D)
      - train_y/test_y: integer labels (N,)
    Returns: trained PyTorch model (nn.Module)
    """
    train_x, train_y, test_x, test_y = dataset

    # Flatten for non-CNN models (so MLP/Softmax accept images too)
    is_cnn = model in {'cnn', 'cnn2', 'Droppcnn', 'Droppcnn2', 'resnet18', 'resnet34'}
    if not is_cnn:
        print('Flattening inputs...')
        if train_x.ndim > 2:
            train_x = train_x.reshape(train_x.shape[0], -1)
        if test_x is not None and getattr(test_x, "ndim", 0) > 2:
            test_x = test_x.reshape(test_x.shape[0], -1)

    n_in = train_x.shape
    n_out = len(np.unique(train_y))

    if batch_size > len(train_y):
        batch_size = len(train_y)

    print(f'Building model with {len(train_x)} training data, {n_out} classes...')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Build net
    if is_cnn:
        if model =='resnet18':
            print('Using a ResNet-18 model...')
            net = CIFARResNet18(n_in=n_in, n_out=n_out)
        elif model == 'resnet34':
            print('Using a ResNet-34 model...')
            net = CIFARResNet34(n_in=n_in, n_out=n_out)
        else:
            print('Using a multilayer convolution neural network based model...')
            net = CNNNet(n_in=n_in, n_hidden=n_hidden, n_out=n_out)
    elif model == 'nn':
        print('Using a multilayer neural network based model...')
        net = MLPNet(n_in=n_in, n_hidden=n_hidden, n_out=n_out)
    else:
        print('Using a single layer softmax based model...')
        net = SoftmaxNet(n_in=n_in, n_out=n_out)

    net.to(device)

    # Loss & optimizer
    criterion = nn.CrossEntropyLoss()
    if model in {'resnet18', 'resnet34'}:
        optimizer = optim.SGD(net.net.parameters(), lr=0.1, momentum=0.9, weight_decay=l2_ratio)
    else:
        optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=l2_ratio)

    # ---- Training ----
    print('Training...')
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for xb_np, yb_np in iterate_minibatches(train_x, train_y, batch_size, shuffle=True):
            xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
            yb = torch.from_numpy(yb_np).to(device=device, dtype=torch.long)

            # --- Multiplicative Gaussian noise (train-time only) ---
            if mg_kappa and mg_kappa > 0.0:
                if mg_mode == "lognorm":
                    # mean-one log-normal: exp(kappa*Z - 0.5*kappa^2)
                    mask = torch.exp(torch.randn_like(xb) * mg_kappa - 0.5 * (mg_kappa ** 2))
                else:
                    # zero-mean multiplicative around 1: 1 + kappa*Z
                    mask = 1.0 + mg_kappa * torch.randn_like(xb)
                xb = xb * mask

            optimizer.zero_grad()
            logits = net(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item())

        if epoch % 10 == 0:
            print(f'Epoch {epoch+1}, train loss {round(running_loss, 3)}')

        # ---- Early-stop check (on clean test set) ----
        if target_test_acc is not None and (epoch % max(1, eval_every) == 0):
            acc = eval_accuracy(net, test_x, test_y, batch_size, device, is_cnn)
            print(f"[eval] epoch {epoch} test acc = {acc*100:.2f}%")
            if acc >= target_test_acc:
                print(f"Reached target test accuracy {target_test_acc*100:.1f}%. Stopping.")
                break

    # ---- optional final test report  ----
    if test_x is not None and test_y is not None and len(test_y) > 0:
        print('Testing...')
        pred_y = _predict_batches(net, test_x, test_y, batch_size, device, is_cnn=is_cnn)
        if len(pred_y) > 0:
            print('Testing Accuracy: {}'.format(accuracy_score(test_y, pred_y)))
            print('More detailed results:')
            print(classification_report(test_y, pred_y))
        else:
            print('No test batches were produced (check batch_size vs dataset size).')

    return net



In [ ]:
# deeplearning.py
%%writefile deeplearning.py

import sys
sys.dont_write_bytecode = True

import numpy as np
import torch

from classifier import train_model, iterate_minibatches  # your PyTorch classifier module

# keep same numpy seed as original for reproducibility of any numpy ops
np.random.seed(21312)
torch.manual_seed(21312)


def _probs_from_batches(model, inputs, batch_size, device):
    """
    Helper: run `model` (nn.Module) on `inputs` (numpy array) in batches (no shuffle),
    return an (N, num_classes) numpy array of softmax probabilities (float32).
    """
    model.eval()
    probs_parts = []
    if batch_size > len(inputs):
        batch_size = len(inputs)
    with torch.no_grad():
        for xb_np, _ in iterate_minibatches(inputs, np.zeros(len(inputs), dtype=np.int32), batch_size, shuffle=False):
            xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
            logits = model(xb)  # model should return logits
            probs = torch.softmax(logits, dim=1)
            probs_parts.append(probs.cpu().numpy())
    if len(probs_parts) == 0:
        return np.zeros((0, 0), dtype=np.float32)
    return np.vstack(probs_parts).astype('float32')


def train_target_model(dataset,
                       epochs=100,
                       batch_size=100,
                       learning_rate=0.01,
                       l2_ratio=1e-7,
                       n_hidden=50,
                       model='nn',
                       # NEW: MG + early-stop controls (all optional)
                       mg_kappa=0.0,          # 0.0 disables multiplicative noise
                       mg_mode='gauss',       # 'gauss' or 'lognorm'
                       target_test_acc=None,  # e.g., 0.35 to stop at 35% test acc
                       eval_every=1):
    """
    Train a target model and build an attack dataset from it.

    Args:
      dataset: tuple (train_x, train_y, test_x, test_y) with numpy arrays.
      epochs, batch_size, learning_rate, l2_ratio, n_hidden, model: passed to train_model.
      mg_kappa: multiplicative Gaussian strength (0 disables).
      mg_mode: 'gauss' (1 + kappa*Z) or 'lognorm' (exp(kappa*Z - 0.5*kappa^2)).
      target_test_acc: early-stop when test accuracy >= this value.
      eval_every: evaluate test accuracy every N epochs for early-stop.

    Returns:
      attack_x: float32 (N_total, num_classes) softmax probs
      attack_y: int32 (N_total,) 1=member (train), 0=non-member (test)
      trained_model: nn.Module
    """
    train_x, train_y, test_x, test_y = dataset

    # Train the model (now forwarding MG + early-stop to classifier.train_model)
    trained_model = train_model(
        (train_x, train_y, test_x, test_y),
        n_hidden=n_hidden,
        batch_size=batch_size,
        epochs=epochs,
        learning_rate=learning_rate,
        model=model,
        l2_ratio=l2_ratio,
        mg_kappa=mg_kappa,
        mg_mode=mg_mode,
        target_test_acc=target_test_acc,
        eval_every=eval_every
    )

    try:
        device = next(trained_model.parameters()).device
    except StopIteration:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # build attack dataset:
    attack_x_parts, attack_y_parts = [], []

    # members (label 1)
    members_probs = _probs_from_batches(trained_model, train_x, batch_size, device)
    if members_probs.size > 0:
        attack_x_parts.append(members_probs)
        attack_y_parts.append(np.ones(len(members_probs), dtype=np.int32))

    # non-members (label 0)
    nonmembers_probs = _probs_from_batches(trained_model, test_x, batch_size, device)
    if nonmembers_probs.size > 0:
        attack_x_parts.append(nonmembers_probs)
        attack_y_parts.append(np.zeros(len(nonmembers_probs), dtype=np.int32))

    if len(attack_x_parts) == 0:
        attack_x = np.zeros((0, 0), dtype=np.float32)
        attack_y = np.zeros((0,), dtype=np.int32)
    else:
        attack_x = np.vstack(attack_x_parts).astype('float32')
        attack_y = np.concatenate(attack_y_parts).astype('int32')

    return attack_x, attack_y, trained_model



In [ ]:
# mlLeaks.py
%%writefile mlLeaks.py
import sys
sys.dont_write_bytecode = True

import os
import pickle
import argparse
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.utils import shuffle as sk_shuffle

from sklearn.metrics import precision_score, recall_score

import deeplearning as dp
import classifier

# -------------------------
# CLI
# -------------------------
parser = argparse.ArgumentParser()
parser.add_argument('--adv', default='1', choices=['1','2','3'], help='Which adversary 1,2,3')
parser.add_argument('--dataset', default='CIFAR10', choices=['CIFAR10','News'], help='Which dataset for target/shadow')
parser.add_argument('--classifierType', default='cnn', choices=['cnn','nn','softmax','resnet18', 'resnet34'], help='Classifier type for dataset1')
parser.add_argument('--dataset2', default='News', choices=['CIFAR10','News'], help='Second dataset for adversary2')
parser.add_argument('--classifierType2', default='nn', choices=['cnn','nn','softmax','resnet18', 'resnet34'], help='Classifier type for dataset2')
parser.add_argument('--dataFolderPath', default='../data', help='Base path to save preprocessed data')
parser.add_argument('--pathToLoadData', default='../data/datasets/cifar-10-batches-py', help='Path to CIFAR pickles')
parser.add_argument('--num_epoch', type=int, default=50, help='Epochs for training shadow/target models')
parser.add_argument('--preprocessData', action='store_true', help='Run preprocessing (otherwise load saved .npz)')
parser.add_argument('--trainTargetModel', action='store_true', help='Train target model (otherwise load saved)')
parser.add_argument('--trainShadowModel', action='store_true', help='Train shadow model (otherwise load saved)')
parser.add_argument('--top_k', type=int, default=3, help='Top-k probabilities to keep for attack features')
parser.add_argument('--attack_clf_sklearn', action='store_true', help='Train attack classifier with sklearn LogisticRegression (default True)')
parser.add_argument('--random_seed', type=int, default=42, help='Random seed')
#### MG #####
parser.add_argument('--trainer', default='standard', choices=['standard','mg','dp'],
                    help='Training mode for target/shadow models')
parser.add_argument('--mg_kappa', type=float, default=0.0,
                    help='Multiplicative Gaussian noise strength (0 disables)')
parser.add_argument('--mg_mode', default='gauss', choices=['gauss','lognorm'],
                    help="MG noise: 'gauss' => (1 + kappa*Z), 'lognorm' => exp(kappa*Z - 0.5*kappa^2)")
parser.add_argument('--target_test_acc', type=float, default=None,
                    help='Early-stop when test accuracy >= this value (e.g., 0.35)')
parser.add_argument('--eval_every', type=int, default=1,
                    help='Evaluate test accuracy every N epochs for early-stop')

#### DP #####
parser.add_argument('--dp_noise_multiplier', type=float, default=1.0,
                    help='DP-SGD noise multiplier (sigma).')
parser.add_argument('--dp_max_grad_norm', type=float, default=1.0,
                    help='Per-sample gradient clip norm C.')
parser.add_argument('--dp_target_epsilon', type=float, default=None,
                    help='Stop early when epsilon <= this (optional).')
parser.add_argument('--dp_target_delta', type=float, default=1e-5,
                    help='Delta for (epsilon, delta)-DP accounting.')
parser.add_argument('--dp_max_epochs', type=int, default=None,
                    help='Optional hard cap on DP epochs (overrides --num_epoch if set).')
# letting shadow use its own trainer; if omitted, it mirrors target
parser.add_argument('--shadow_trainer', default=None, choices=[None,'standard','mg','dp'],
                    help='If set, overrides --trainer for the shadow model.')

#opt = parser.parse_args()
opt, _ = parser.parse_known_args()


np.random.seed(opt.random_seed)
random.seed(opt.random_seed)
torch.manual_seed(opt.random_seed)


def clip_top_k(data, top=3):
    """Keep only the top-k values of each row (descending). Returns (N,top)."""
    if top is None:
        return data
    # data: (N, C) or (N, >=1)
    res = [sorted(row, reverse=True)[:top] for row in data]
    return np.array(res, dtype=np.float32)

def read_cifar10(data_path):
    """Load original CIFAR-10 Python pickles (works in Python3). Returns X (N,3072), y (N,)."""
    X_parts = []
    y_parts = []
    for i in range(1, 6):
        p = Path(data_path) / f"data_batch_{i}"
        with open(p, 'rb') as f:
            batch = pickle.load(f, encoding='latin1')
        X_parts.append(np.array(batch['data']))
        y_parts.append(np.array(batch['labels']))
    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    # test batch
    with open(Path(data_path) / 'test_batch', 'rb') as f:
        test_batch = pickle.load(f, encoding='latin1')
    Xtest = np.array(test_batch['data'])
    ytest = np.array(test_batch['labels'])
    return X, y, Xtest, ytest

def reshape_and_normalize_cifar(train_flat, test_flat):
    """From (N,3072) -> (N,3,32,32) channels-first, normalized by train mean/std."""
    def reshape(raw):
        raw = np.dstack((raw[:, :1024], raw[:, 1024:2048], raw[:, 2048:]))
        raw = raw.reshape((raw.shape[0], 32, 32, 3)).transpose(0,3,1,2)
        return raw.astype(np.float32)
    train_img = reshape(train_flat)
    test_img = reshape(test_flat)
    mean = np.mean(train_img, axis=0)
    std = np.std(train_img, axis=0).clip(min=1.0)
    train_scaled = (train_img - mean) / std
    test_scaled = (test_img - mean) / std
    return train_scaled.astype(np.float32), test_scaled.astype(np.float32)

def preprocess_news(all_texts_train, all_texts_test, max_features=None):
    """TF-IDF vectorize (train+test combined to share vocab), then normalize columns."""
    vectorizer = TfidfVectorizer(max_features=max_features)
    combined = np.concatenate([all_texts_train, all_texts_test], axis=0)
    X = vectorizer.fit_transform(combined).toarray()
    # split back
    n_train = len(all_texts_train)
    train = X[:n_train]
    test = X[n_train:]
    # normalize
    mean = np.mean(train, axis=0)
    std = np.std(train, axis=0).clip(min=1.0)
    return ((train - mean) / std).astype(np.float32), ((test - mean) / std).astype(np.float32)

def list_shuffle_split(X, y, cluster):
    """Shuffle (stable) and split into 4 groups each size cluster.
       Returns 8 arrays: toTrain, toTrainLabel, shadowTrain, shadowTrainLabel, toTest, toTestLabel, shadowTest, shadowTestLabel
    """
    arr = list(zip(X, y))
    random.shuffle(arr)
    Xs, ys = zip(*arr)
    Xs = np.array(Xs)
    ys = np.array(ys)
    # ensure enough length
    total_needed = cluster * 4
    if len(Xs) < total_needed:
        raise ValueError(f"Not enough data for cluster={cluster}; need {total_needed}, got {len(Xs)}")
    a = Xs
    b = ys
    toTrainData, toTrainLabel = a[:cluster], b[:cluster]
    shadowData, shadowLabel = a[cluster:2*cluster], b[cluster:2*cluster]
    toTestData, toTestLabel = a[2*cluster:3*cluster], b[2*cluster:3*cluster]
    shadowTestData, shadowTestLabel = a[3*cluster:4*cluster], b[3*cluster:4*cluster]
    return toTrainData, toTrainLabel, shadowData, shadowLabel, toTestData, toTestLabel, shadowTestData, shadowTestLabel

def save_npz(path, *arrays):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, *arrays)

def load_npz(path):
    with np.load(path) as f:
        return [f[f"arr_{i}"] for i in range(len(f.files))]


def initialize_data(dataset, origin_path, data_folder='./data/'):
    data_folder = Path(data_folder)
    path_out = data_folder / dataset / 'Preprocessed'
    path_out.mkdir(parents=True, exist_ok=True)

    if dataset == 'CIFAR10':
        print("Loading CIFAR-10 from", origin_path)
        X, y, Xtest, ytest = read_cifar10(origin_path)
        cluster = 10520
        # we combine both train+test from official into a single pool like original code did
        X_all = np.concatenate([X, Xtest], axis=0)
        y_all = np.concatenate([y, ytest], axis=0)
        toTrain, toTrainLabel, shadow, shadowLabel, toTest, toTestLabel, shadowTest, shadowTestLabel = list_shuffle_split(X_all, y_all, cluster)
        toTrainSave, toTestSave = reshape_and_normalize_cifar(toTrain, toTest)
        shadowSave, shadowTestSave = reshape_and_normalize_cifar(shadow, shadowTest)
    else:  # News
        from sklearn.datasets import fetch_20newsgroups
        print("Fetching 20 newsgroups")
        newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
        newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers','footers','quotes'))
        texts_train = newsgroups_train.data
        texts_test = newsgroups_test.data
        labels_train = newsgroups_train.target
        labels_test = newsgroups_test.target
        X_all_texts = np.concatenate([texts_train, texts_test], axis=0)
        y_all = np.concatenate([labels_train, labels_test], axis=0)
        cluster = 4500
        toTrain, toTrainLabel, shadow, shadowLabel, toTest, toTestLabel, shadowTest, shadowTestLabel = list_shuffle_split(X_all_texts, y_all, cluster)
        # vectorize using TF-IDF on the union of train/test for stable vocab
        toTrainSave, toTestSave = preprocess_news(toTrain, toTest, max_features=None)
        shadowSave, shadowTestSave = preprocess_news(shadow, shadowTest, max_features=None)


    save_npz(str(path_out / 'targetTrain.npz'), toTrainSave, toTrainLabel)
    save_npz(str(path_out / 'targetTest.npz'), toTestSave, toTestLabel)
    save_npz(str(path_out / 'shadowTrain.npz'), shadowSave, shadowLabel)
    save_npz(str(path_out / 'shadowTest.npz'), shadowTestSave, shadowTestLabel)
    print("Saved preprocessed data to:", path_out)

# -------------------------
# Train + save target/shadow models + build attack datasets
# -------------------------
def initialize_target_model(dataset, num_epoch, data_folder='./data/', model_folder='./model/', classifier_type='cnn', batch_size=100, learning_rate=1e-3, top_k=3):
    data_path = Path(data_folder) / dataset / 'Preprocessed'
    attacker_out = Path(data_folder) / dataset / 'attackerModelData'
    model_out = Path(model_folder) / dataset
    attacker_out.mkdir(parents=True, exist_ok=True)
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"Training target model for {num_epoch} epochs...")
    target_train, target_train_label = load_npz(str(data_path / 'targetTrain.npz'))
    target_test, target_test_label = load_npz(str(data_path / 'targetTest.npz'))

    # # train_target_model returns attack_x, attack_y, trained_model
    # attack_x, attack_y, model = dp.train_target_model(
    #     dataset=(target_train.astype(np.float32), target_train_label.astype(np.int32),
    #              target_test.astype(np.float32), target_test_label.astype(np.int32)),
    #     epochs=num_epoch, batch_size=batch_size, learning_rate=learning_rate, n_hidden=8192, model=classifier_type
    # )
    # #n_hidden=128

    attack_x, attack_y, model = dp.train_target_model(
    dataset=(target_train.astype(np.float32), target_train_label.astype(np.int32),
             target_test.astype(np.float32),  target_test_label.astype(np.int32)),
    epochs=num_epoch,
    batch_size=batch_size,
    learning_rate=learning_rate,
    l2_ratio=1e-7,
    n_hidden=100,
    model=classifier_type,
    # --- pass MG + early-stop flags through ---
    mg_kappa=opt.mg_kappa if opt.trainer == 'mg' else 0.0,
    mg_mode=opt.mg_mode,
    target_test_acc=opt.target_test_acc,
    eval_every=opt.eval_every
    )


    # save attack data and model state_dict
    attack_x = attack_x.astype(np.float32)
    attack_y = attack_y.astype(np.int32)
    save_npz(str(attacker_out / 'targetModelData.npz'), attack_x, attack_y)
    # save PyTorch model
    try:
        torch.save(model.state_dict(), str(model_out / 'targetModel.pth'))
    except Exception as e:
        print("Warning: unable to save model state_dict:", e)

    # optionally clip top-k later in pipeline
    return attack_x, attack_y, model

def initialize_shadow_model(dataset, num_epoch, data_folder='./data/', model_folder='./model/', classifier_type='cnn', batch_size=100, learning_rate=1e-3, top_k=3):
    data_path = Path(data_folder) / dataset / 'Preprocessed'
    attacker_out = Path(data_folder) / dataset / 'attackerModelData'
    model_out = Path(model_folder) / dataset
    attacker_out.mkdir(parents=True, exist_ok=True)
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"Training shadow model for {num_epoch} epochs...")
    shadow_train, shadow_train_label = load_npz(str(data_path / 'shadowTrain.npz'))
    shadow_test, shadow_test_label = load_npz(str(data_path / 'shadowTest.npz'))

    # attack_x, attack_y, model = dp.train_target_model(
    #     dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
    #              shadow_test.astype(np.float32), shadow_test_label.astype(np.int32)),
    #     epochs=num_epoch, batch_size=batch_size, learning_rate=learning_rate, n_hidden=8192, model=classifier_type
    # )

    ######## attack option A: Make shadow mimic the target
    attack_x, attack_y, model = dp.train_target_model(
    dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
             shadow_test.astype(np.float32),  shadow_test_label.astype(np.int32)),
    epochs=num_epoch,
    batch_size=batch_size,
    learning_rate=learning_rate,
    l2_ratio=1e-7,
    n_hidden=100,
    model=classifier_type,
    mg_kappa=opt.mg_kappa if opt.trainer == 'mg' else 0.0,
    mg_mode=opt.mg_mode,
    target_test_acc=opt.target_test_acc,
    eval_every=opt.eval_every
    )

    # ###### Attack option B: Keep shadow standard (no MG), while target uses MG
    # attack_x, attack_y, model = dp.train_target_model(
    # dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
    #          shadow_test.astype(np.float32),  shadow_test_label.astype(np.int32)),
    # epochs=num_epoch,
    # batch_size=batch_size,
    # learning_rate=learning_rate,
    # l2_ratio=1e-7,
    # n_hidden=8192,
    # model=classifier_type,
    # mg_kappa=0.0,                 # <- force off for shadow
    # mg_mode=opt.mg_mode,
    # target_test_acc=opt.target_test_acc,
    # eval_every=opt.eval_every
    # )


    attack_x = attack_x.astype(np.float32)
    attack_y = attack_y.astype(np.int32)
    save_npz(str(attacker_out / 'shadowModelData.npz'), attack_x, attack_y)
    try:
        torch.save(model.state_dict(), str(model_out / 'shadowModel.pth'))
    except Exception as e:
        print("Warning: unable to save model state_dict:", e)

    return attack_x, attack_y, model

# -------------------------
# Load precomputed attack data (and optionally model)
# -------------------------
def load_attack_data(dataset, kind='target', data_folder='./data/'):
    data_path = Path(data_folder) / dataset / 'attackerModelData'
    arr = load_npz(str(data_path / f'{kind}ModelData.npz'))
    return arr[0].astype(np.float32), arr[1].astype(np.int32)

# -------------------------
# Attack classifier training + evaluation
# -------------------------
def train_attack_classifier_and_eval(train_X, train_y, test_X, test_y, balance=True):
    """Train sklearn LogisticRegression on train_X/train_y and evaluate on test_X/test_y."""
    # optionally balance (downsample majority)
    if balance:
        pos = np.where(train_y == 1)[0]
        neg = np.where(train_y == 0)[0]
        m = min(len(pos), len(neg))
        if m == 0:
            raise ValueError("One of classes empty in attack training data.")
        sel = np.concatenate([np.random.choice(pos, m, replace=False), np.random.choice(neg, m, replace=False)])
        train_X_bal, train_y_bal = train_X[sel], train_y[sel]
    else:
        train_X_bal, train_y_bal = train_X, train_y

    train_X_bal, train_y_bal = sk_shuffle(train_X_bal, train_y_bal, random_state=opt.random_seed)

    clf = LogisticRegression(max_iter=2000, solver='lbfgs')
    clf.fit(train_X_bal, train_y_bal)

    preds = clf.predict(test_X)
    probs = clf.predict_proba(test_X)[:, 1] if hasattr(clf, "predict_proba") else None

    acc = accuracy_score(test_y, preds)
    auc = roc_auc_score(test_y, probs) if probs is not None else None
    print("Attack classifier results — accuracy: {:.4f} AUC: {}".format(acc, auc))
    print(classification_report(test_y, preds))

    preds = clf.predict(test_X)
    probs = clf.predict_proba(test_X)[:, 1] if hasattr(clf, "predict_proba") else None

    acc = accuracy_score(test_y, preds)
    auc = roc_auc_score(test_y, probs) if probs is not None else None
    prec = precision_score(test_y, preds, average="binary", zero_division=0)
    rec  = recall_score(test_y, preds,    average="binary", zero_division=0)

    print("Attack classifier — acc: {:.4f}  AUC: {}  Prec: {:.4f}  Rec: {:.4f}".format(acc, auc, prec, rec))
    print(classification_report(test_y, preds))


    ####
    return clf, {"acc": acc, "auc": auc, "prec": prec, "rec": rec}


# ----------------------------------------------------------------
# Orchestration: build/generate attack data, run attacker
# ----------------------------------------------------------------
def generate_attack_data(dataset, classifierType, dataFolderPath, pathToLoadData, num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=3):
    attackerModelDataPath = Path(dataFolderPath) / dataset / 'attackerModelData'
    # Preprocess (once)
    if preprocessData:
        initialize_data(dataset, pathToLoadData, data_folder=dataFolderPath)

    # target
    if trainTargetModel:
        tX, tY, tmodel = initialize_target_model(dataset, num_epoch, data_folder=dataFolderPath, classifier_type=classifierType)
    else:
        tX, tY = load_attack_data(dataset, 'target', data_folder=dataFolderPath)
        tmodel = None

    # shadow
    if trainShadowModel:
        sX, sY, smodel = initialize_shadow_model(dataset, num_epoch, data_folder=dataFolderPath, classifier_type=classifierType)
    else:
        sX, sY = load_attack_data(dataset, 'shadow', data_folder=dataFolderPath)
        smodel = None

    # clip top-k
    tX_clipped = clip_top_k(tX, top=top_k)
    sX_clipped = clip_top_k(sX, top=top_k)
    return tX_clipped, tY, sX_clipped, sY, tmodel, smodel


def attacker_one(dataset='CIFAR10', classifierType='cnn', dataFolderPath='../data', pathToLoadData='../data/datasets/cifar-10-batches-py', num_epoch=50, preprocessData=True, trainTargetModel=True, trainShadowModel=True, top_k=3):
    tX, tY, sX, sY, tmodel, smodel = generate_attack_data(dataset, classifierType, dataFolderPath, pathToLoadData, num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=top_k)
    print("Training attack classifier (train on SHADOW, evaluate on TARGET).")
    # IMPORTANT: train on shadow (sX,sY), test on target (tX,tY)
    clf, acc, auc = train_attack_classifier_and_eval(sX, sY, tX, tY, balance=True)
    return clf, acc, auc

def attacker_two(dataset1='CIFAR10', dataset2='News', classifierType1='cnn', classifierType2='nn', dataFolderPath='./data/', pathToLoadData='./data/cifar-10-batches-py-official', num_epoch=50, preprocessData=True, trainTargetModel=True, trainShadowModel=True, top_k=3):
    # shadow from dataset1, target from dataset2 (or vice versa depending on experiment)
    print("Generating attack data for dataset1 (shadow) and dataset2 (target).")
    _, _, sX, sY, _, _ = generate_attack_data(dataset1, classifierType1, dataFolderPath, pathToLoadData, num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=top_k)
    tX, tY, _, _, tmodel, _ = generate_attack_data(dataset2, classifierType2, dataFolderPath, pathToLoadData, num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=top_k)
    print("Training attack classifier (train on SHADOW from dataset1, evaluate on TARGET from dataset2).")
    clf, acc, auc = train_attack_classifier_and_eval(sX, sY, tX, tY, balance=True)
    return clf, acc, auc

def attacker_three(dataset='CIFAR10', classifierType='cnn', dataFolderPath='./data/', pathToLoadData='./data/cifar-10-batches-py-official', num_epoch=50, preprocessData=True, trainTargetModel=True, top_k=1):
    # Compute AUC of top-1 probability as a simple detector on the target model (diagnostic)
    tX, tY, _, _, tmodel, _ = generate_attack_data(dataset, classifierType, dataFolderPath, pathToLoadData, num_epoch, preprocessData, trainTargetModel, trainShadowModel=False, top_k=top_k)
    # tX shape (N, 1) if top_k==1
    if tX.ndim == 2 and tX.shape[1] == 1:
        scores = tX.squeeze()
        auc = roc_auc_score(tY, scores)
        print(f"AUC of top-1 probability on target model (diagnostic) = {auc:.4f}")
        return auc
    else:
        raise ValueError("attacker_three expects top_k=1 to compute AUC of single score per sample.")

def attack_epochs_curve(
    epochs_list=(10,20,30,40,50,60,80,100),
    dataset="CIFAR10",
    classifierType="cnn",
    dataFolderPath="../data",
    batch_size=100,
    learning_rate=1e-3,
    n_hidden=100,
    top_k=3,
    trainer="mg",          # "mg" or "standard"
    mg_kappa=0.0,
    mg_mode="gauss",
):
    """
    For each E in epochs_list:
      - train TARGET for E epochs
      - train SHADOW for E epochs
      - build attack data (shadow->train, target->test), train attack
      - collect precision/recall
    Returns: dict {"epochs": [...], "precision": [...], "recall": [...]}
    """
    import numpy as np
    from pathlib import Path

    # Load preprocessed splits once
    data_path = Path(dataFolderPath) / dataset / 'Preprocessed'
    target_train, target_train_label = load_npz(str(data_path / 'targetTrain.npz'))
    target_test,  target_test_label  = load_npz(str(data_path / 'targetTest.npz'))
    shadow_train, shadow_train_label = load_npz(str(data_path / 'shadowTrain.npz'))
    shadow_test,  shadow_test_label  = load_npz(str(data_path / 'shadowTest.npz'))

    xs, precisions, recalls = [], [], []

    for E in epochs_list:
        print(f"\n=== Epoch budget: {E} ===")

        # Train TARGET
        tX, tY, _tmodel = dp.train_target_model(
            dataset=(target_train.astype(np.float32), target_train_label.astype(np.int32),
                     target_test.astype(np.float32),  target_test_label.astype(np.int32)),
            epochs=E, batch_size=batch_size, learning_rate=learning_rate,
            l2_ratio=1e-7, n_hidden=n_hidden, model=classifierType,
            mg_kappa=(mg_kappa if trainer == "mg" else 0.0),
            mg_mode=mg_mode, target_test_acc=None, eval_every=1
        )

        # Train SHADOW
        sX, sY, _smodel = dp.train_target_model(
            dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
                     shadow_test.astype(np.float32),  shadow_test_label.astype(np.int32)),
            epochs=E, batch_size=batch_size, learning_rate=learning_rate,
            l2_ratio=1e-7, n_hidden=n_hidden, model=classifierType,
            mg_kappa=(mg_kappa if trainer == "mg" else 0.0),
            mg_mode=mg_mode, target_test_acc=None, eval_every=1
        )

        # top-k clip like ML-Leaks
        sXc = clip_top_k(sX, top=top_k)
        tXc = clip_top_k(tX, top=top_k)

        # Train attack on SHADOW, test on TARGET
        _, metrics = train_attack_classifier_and_eval(sXc, sY, tXc, tY, balance=True)

        xs.append(E)
        precisions.append(metrics["prec"])
        recalls.append(metrics["rec"])

    return {"epochs": xs, "precision": precisions, "recall": recalls}


if __name__ == "__main__":
    if opt.adv == '1':
        attacker_one(dataset=opt.dataset, classifierType=opt.classifierType, dataFolderPath=opt.dataFolderPath,
                     pathToLoadData=opt.pathToLoadData, num_epoch=opt.num_epoch, preprocessData=opt.preprocessData,
                     trainTargetModel=opt.trainTargetModel, trainShadowModel=opt.trainShadowModel, top_k=opt.top_k)
    elif opt.adv == '2':
        attacker_two(dataset1=opt.dataset, dataset2=opt.dataset2,
                     classifierType1=opt.classifierType, classifierType2=opt.classifierType2,
                     dataFolderPath=opt.dataFolderPath, pathToLoadData=opt.pathToLoadData,
                     num_epoch=opt.num_epoch, preprocessData=opt.preprocessData,
                     trainTargetModel=opt.trainTargetModel, trainShadowModel=opt.trainShadowModel, top_k=opt.top_k)
    elif opt.adv == '3':
        attacker_three(dataset=opt.dataset, classifierType=opt.classifierType, dataFolderPath=opt.dataFolderPath,
                       pathToLoadData=opt.pathToLoadData, num_epoch=opt.num_epoch, preprocessData=opt.preprocessData,
                       trainTargetModel=opt.trainTargetModel, top_k=opt.top_k)


In [ ]:
# # visualize.py
# %%writefile visualize.py
# import argparse
# import matplotlib.pyplot as plt

# from mlLeaks import attack_epochs_curve

# def parse_list_floats(s: str):
#     return [float(x.strip()) for x in s.split(",") if x.strip()]

# def parse_list_ints(s: str):
#     return [int(x.strip()) for x in s.split(",") if x.strip()]

# def main():
#     p = argparse.ArgumentParser(description="Plot attack precision/recall vs epochs for multiple kappa values.")
#     p.add_argument("--kappas", type=str, default="0.0,0.25,0.5", help="Comma-separated κ values")
#     p.add_argument("--epochs", type=str, default="10,20,30,40,50,60,80,100", help="Comma-separated epoch budgets")
#     p.add_argument("--dataset", default="CIFAR10", choices=["CIFAR10","News"])
#     p.add_argument("--classifierType", default="cnn", choices=["cnn","nn","softmax","resnet18","resnet34"])
#     p.add_argument("--dataFolderPath", default="../data")
#     p.add_argument("--batch_size", type=int, default=100)
#     p.add_argument("--learning_rate", type=float, default=1e-3)
#     p.add_argument("--n_hidden", type=int, default=100)
#     p.add_argument("--top_k", type=int, default=3)
#     p.add_argument("--mg_mode", default="gauss", choices=["gauss","lognorm"])
#     p.add_argument("--trainer", default="mg", choices=["standard","mg"])
#     p.add_argument("--out", default="attack_curve_precision_recall_multi_kappa.png")
#     p.add_argument("--ylim_lo", type=float, default=0.4)
#     p.add_argument("--ylim_hi", type=float, default=1.1)
#     p.add_argument("--legend_loc", default="upper right")
#     #p.add_argument("--wandb_log", action="store_true")
#     args = p.parse_args()

#     kappas = parse_list_floats(args.kappas)
#     epochs_list = parse_list_ints(args.epochs)

#     curves, labels = [], []
#     for kappa in kappas:
#         print(f"\n===== κ = {kappa} =====")
#         curve = attack_epochs_curve(
#             epochs_list=epochs_list,
#             dataset=args.dataset,
#             classifierType=args.classifierType,
#             dataFolderPath=args.dataFolderPath,
#             batch_size=args.batch_size,
#             learning_rate=args.learning_rate,
#             n_hidden=args.n_hidden,
#             top_k=args.top_k,
#             trainer=("mg" if args.trainer == "mg" else "standard"),
#             mg_kappa=(kappa if args.trainer == "mg" else 0.0),
#             mg_mode=args.mg_mode,
#         )
#         curves.append(curve)
#         labels.append(f"κ={kappa}" if args.trainer == "mg" else "Standard")
#   # ###############################
#   #   # Plot: two panels
#   #   plt.figure(figsize=(9,3.2))

#   #   ax1 = plt.subplot(1,2,1)
#   #   for curve, label in zip(curves, labels):
#   #       ax1.plot(curve["epochs"], curve["precision"], marker="o", linewidth=2, markersize=4, label=label)
#   #   ax1.set_xlabel("Number Of Epochs")
#   #   ax1.set_ylabel("Attack Precision")
#   #   ax1.set_ylim(args.ylim_lo, args.ylim_hi)
#   #   ax1.grid(True, alpha=0.3)
#   #   ax1.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, fontsize=8)
#   #   ax1.set_title("(a) Precision")

#   #   ax2 = plt.subplot(1,2,2)
#   #   for curve, label in zip(curves, labels):
#   #       ax2.plot(curve["epochs"], curve["recall"], marker="s", linewidth=2, markersize=4, label=label)
#   #   ax2.set_xlabel("Number Of Epochs")
#   #   ax2.set_ylabel("Attack Recall")
#   #   ax2.set_ylim(args.ylim_lo, args.ylim_hi)
#   #   ax2.grid(True, alpha=0.3)
#   #   ax2.set_title("(b) Recall")

#   #   plt.tight_layout()
#   #   plt.savefig(args.out, dpi=300, bbox_inches="tight")
#   #   print("Saved:", args.out)

#   #   #################
#     ###############################
#     # Make two separate figures (precision / recall)
#     # Legend inside the axes at the top-right (no bbox_to_anchor).

#     # --- Precision ---
#     fig1, ax1 = plt.subplots(figsize=(5, 3.6))
#     for curve, label in zip(curves, labels):
#         ax1.plot(curve["epochs"], curve["precision"], marker="o",
#                 linewidth=2, markersize=4, label=label)
#     ax1.set_xlabel("Number Of Epochs")
#     ax1.set_ylabel("Attack Precision")
#     ax1.set_ylim(args.ylim_lo, args.ylim_hi)
#     ax1.grid(True, alpha=0.3)
#     ax1.set_title("(a) Precision")
#     ax1.legend(loc=args.legend_loc, fontsize=9, frameon=True)

#     fig1.tight_layout()
#     out_prec = args.out.replace(".png", "_precision.png")
#     fig1.savefig(out_prec, dpi=300, bbox_inches="tight")
#     print("Saved:", out_prec)

#     # --- Recall ---
#     fig2, ax2 = plt.subplots(figsize=(5, 3.6))
#     for curve, label in zip(curves, labels):
#         ax2.plot(curve["epochs"], curve["recall"], marker="s",
#                 linewidth=2, markersize=4, label=label)
#     ax2.set_xlabel("Number Of Epochs")
#     ax2.set_ylabel("Attack Recall")
#     ax2.set_ylim(args.ylim_lo, args.ylim_hi)
#     ax2.grid(True, alpha=0.3)
#     ax2.set_title("(b) Recall")
#     ax2.legend(loc=args.legend_loc, fontsize=9, frameon=True)

#     fig2.tight_layout()
#     out_rec = args.out.replace(".png", "_recall.png")
#     fig2.savefig(out_rec, dpi=300, bbox_inches="tight")
#     print("Saved:", out_rec)


# if __name__ == "__main__":
#     main()


In [ ]:
%%writefile visualize.py
import argparse
import os
import matplotlib.pyplot as plt

from mlLeaks import attack_epochs_curve


def parse_list_floats(s: str):
    return [float(x.strip()) for x in s.split(",") if x.strip()]


def parse_list_ints(s: str):
    return [int(x.strip()) for x in s.split(",") if x.strip()]


def _ensure_dir(path: str):
    parent = os.path.dirname(path) or "."
    os.makedirs(parent, exist_ok=True)


def _with_suffix_png(base_path: str, suffix: str) -> str:
    """
    Returns base_path with _{suffix}.png appended.
    If base_path already ends with .png, insert before the extension.
    """
    base_abs = os.path.abspath(base_path)
    if base_abs.lower().endswith(".png"):
        return base_abs[:-4] + f"_{suffix}.png"
    return base_abs + f"_{suffix}.png"


def main():
    p = argparse.ArgumentParser(
        description="Plot attack precision/recall vs epochs for multiple kappa values."
    )
    p.add_argument("--kappas", type=str, default="0.0,0.25,0.5", help="Comma-separated κ values")
    p.add_argument("--epochs", type=str, default="10,20,30,40,50,60,80,100", help="Comma-separated epoch budgets")
    p.add_argument("--dataset", default="CIFAR10", choices=["CIFAR10", "News"])
    p.add_argument("--classifierType", default="cnn", choices=["cnn", "nn", "softmax", "resnet18", "resnet34"])
    p.add_argument("--dataFolderPath", default="../data")
    p.add_argument("--batch_size", type=int, default=100)
    p.add_argument("--learning_rate", type=float, default=1e-3)
    p.add_argument("--n_hidden", type=int, default=100)
    p.add_argument("--top_k", type=int, default=3)
    p.add_argument("--mg_mode", default="gauss", choices=["gauss", "lognorm"])
    p.add_argument("--trainer", default="mg", choices=["standard", "mg"])
    p.add_argument("--out", default="attack_curve_precision_recall_multi_kappa.png",
                   help="Base output path (precision/recall files will be derived from this)")
    p.add_argument("--ylim_lo", type=float, default=0.3)
    p.add_argument("--ylim_hi", type=float, default=1.1)
    p.add_argument("--legend_loc", default="upper right")
    args = p.parse_args()

    kappas = parse_list_floats(args.kappas)
    epochs_list = parse_list_ints(args.epochs)

    curves, labels = [], []
    for kappa in kappas:
        print(f"\n===== κ = {kappa} =====")
        curve = attack_epochs_curve(
            epochs_list=epochs_list,
            dataset=args.dataset,
            classifierType=args.classifierType,
            dataFolderPath=args.dataFolderPath,
            batch_size=args.batch_size,
            learning_rate=args.learning_rate,
            n_hidden=args.n_hidden,
            top_k=args.top_k,
            trainer=("mg" if args.trainer == "mg" else "standard"),
            mg_kappa=(kappa if args.trainer == "mg" else 0.0),
            mg_mode=args.mg_mode,
        )
        curves.append(curve)
        labels.append(f"κ={kappa}" if args.trainer == "mg" else "Standard")

    # ---------- Save two separate figures (legends inside, top-right) ----------
    out_prec = _with_suffix_png(args.out, "precision")
    out_rec = _with_suffix_png(args.out, "recall")
    _ensure_dir(out_prec)
    _ensure_dir(out_rec)

    # Precision figure
    fig1, ax1 = plt.subplots(figsize=(5, 3.6))
    for curve, label in zip(curves, labels):
        ax1.plot(curve["epochs"], curve["precision"], marker="o", linewidth=2, markersize=4, label=label)
    ax1.set_xlabel("Number Of Epochs")
    ax1.set_ylabel("Attack Precision")
    ax1.set_ylim(args.ylim_lo, args.ylim_hi)
    ax1.grid(True, alpha=0.3)
    ax1.set_title("(a) Precision")
    ax1.legend(loc=args.legend_loc, fontsize=9, frameon=True)
    fig1.tight_layout()
    fig1.savefig(out_prec, dpi=300, bbox_inches="tight")
    print("Saved:", out_prec)

    # Recall figure
    fig2, ax2 = plt.subplots(figsize=(5, 3.6))
    for curve, label in zip(curves, labels):
        ax2.plot(curve["epochs"], curve["recall"], marker="s", linewidth=2, markersize=4, label=label)
    ax2.set_xlabel("Number Of Epochs")
    ax2.set_ylabel("Attack Recall")
    ax2.set_ylim(args.ylim_lo, args.ylim_hi)
    ax2.grid(True, alpha=0.3)
    ax2.set_title("(b) Recall")
    ax2.legend(loc=args.legend_loc, fontsize=9, frameon=True)
    fig2.tight_layout()
    fig2.savefig(out_rec, dpi=300, bbox_inches="tight")
    print("Saved:", out_rec)


if __name__ == "__main__":
    main()


In [ ]:
# !python visualize.py \
#   --kappas "0.0,0.5,0.75,1.25" \
#   --epochs "10,20,30,40,50,60,80,100" \
#   --dataset CIFAR10 \
#   --classifierType cnn \
#   --dataFolderPath ../data \
#   --trainer mg \
#   --mg_mode gauss \
#   --top_k 3 \
#   --out attack_curve_precision_recall_multi_kappa2.png

!python visualize.py \
  --kappas "0.0,0.5,1.2,1.8" \
  --epochs "20,40,60,80,100,120" \
  --dataset CIFAR10 \
  --classifierType cnn \
  --dataFolderPath ../data \
  --trainer mg \
  --mg_mode gauss \
  --top_k 3 \
  --out ../figures/attack_curve_nn_precision_recall_multi_kappa.png


In [ ]:
from IPython.display import Image, display
display(Image(filename="../figures/attack_curve_nn_precision_recall_multi_kappa_precision.png"))
display(Image(filename="../figures/attack_curve_nn_precision_recall_multi_kappa_recall.png"))